# Module SC27 - Detector RE-EVALUATION on the expanded test split (no retraining)

> **Run on Kaggle** - Settings: Accelerator **GPU T4 x2**, Internet **ON**. Attach inputs below,
> then **Save Version (Save & Run All)**. Runtime ~5-10 minutes (inference only).

The deployed YOLO11 detector (Module SC8) is deliberately kept UNCHANGED after the dataset
expansion. Its published metrics, however, were measured on the old 46-item test split. This
module re-evaluates the SAME fine-tuned weights on the CURRENT dataset's test split (50 items,
including the new hazardous_chemical/composite items), so the report can cite detector metrics
on the same test set as every classification result. Nothing is trained; if no fine-tuned
single-class detector is attached the module STOPS rather than training a new one.

**Inputs:** `thesis-common-10c`, the image dataset (current 485-item version), Module-SC1 output
(`config.json`), and the **Module-SC8 output** (its `det/weights/best.pt` fine-tuned detector).

**Outputs:** `detector_reeval.json` (mAP@50, mAP@50-95, precision, recall, per-image detect rate,
counts + weights provenance) and `detector_reeval.csv` (one-row table).

In [ ]:
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS))

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'], check=False)
from ultralytics import YOLO

ROOT = T.find_image_root()
df_tr, df_va, df_te = T.load_manifests(ROOT); T.check_leakage(df_tr, df_va, df_te)
n_items = df_te.item_id.nunique(); n_views = len(df_te[df_te.source=='original']) if 'source' in df_te else len(df_te)
print(f'test split: {n_items} items, {n_views} original views')

# export the CURRENT dataset in YOLO format (original rows only; includes the test split + data.yaml)
DATA_YAML = T.export_yolo_dataset(ROOT, '/kaggle/working/yolo_ds', include_aug=False)

# ---- locate the FINE-TUNED single-class detector (same discovery rule as SC18) --------------
# A file literally named 'yolo11s.pt' is usually the COCO-pretrained download (80 classes),
# NOT the fine-tuned model - accept only a checkpoint whose class list is the single waste class.
def is_waste_detector(path):
    try:
        m = YOLO(path)
        names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
        return len(names) == 1, m
    except Exception:
        return False, None

det, det_path = None, None
cands  = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True), key=len)
cands += sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True), key=len)
seen = set()
for p in cands:
    if p in seen or os.path.basename(p).startswith('weights_'): continue
    seen.add(p)
    ok, m = is_waste_detector(p)
    if ok:
        det, det_path = m, p; print('loaded FINE-TUNED detector:', p); break
assert det is not None, ('No fine-tuned single-class detector found in the inputs - attach the '
                         'Module-SC8 output (det/weights/best.pt). This module never trains.')

### Evaluate on the expanded test split + per-view detect rate

In [ ]:
# ---- evaluate on the CURRENT test split ----------------------------------------------------
res = det.val(data=DATA_YAML, split='test', imgsz=640, verbose=False)
mAP50, mAP5095 = float(res.box.map50), float(res.box.map)
P, R = float(res.box.mp), float(res.box.mr)
print(f'TEST  (expanded split): mAP@50={mAP50:.3f}  mAP@50-95={mAP5095:.3f}  P={P:.3f}  R={R:.3f}')

# per-image detect rate at the deployment confidence (does at least one box fire on every view?)
import glob as g
test_imgs = sorted(g.glob('/kaggle/working/yolo_ds/images/test/*'))
missed = []
for chunk in range(0, len(test_imgs), 64):
    for r in det.predict(test_imgs[chunk:chunk+64], conf=0.25, verbose=False):
        if len(r.boxes) == 0: missed.append(os.path.basename(r.path))
detect_rate = 1 - len(missed)/max(1,len(test_imgs))
print(f'detect rate @conf0.25: {detect_rate:.4f}  ({len(test_imgs)-len(missed)}/{len(test_imgs)} views; missed={missed[:5]})')

### Save results (with the old published numbers embedded for the report comparison)

In [ ]:
# ---- save + compare against the previously published (46-item test) numbers ----------------
OLD = {'mAP50': 0.991, 'mAP50_95': 0.850, 'precision': 0.980, 'recall': 0.981,
       'note': 'published numbers, evaluated on the initial 457-item release (46-item test)'}
out = {'weights': det_path,
       'test_items': int(n_items), 'test_views': len(test_imgs),
       'mAP50': round(mAP50,4), 'mAP50_95': round(mAP5095,4),
       'precision': round(P,4), 'recall': round(R,4),
       'detect_rate_conf025': round(detect_rate,4), 'missed_views': missed,
       'previous_published': OLD}
json.dump(out, open('detector_reeval.json','w'), indent=2)
import pandas as pd
pd.DataFrame([{k:v for k,v in out.items() if not isinstance(v,(dict,list))}]).to_csv('detector_reeval.csv', index=False)
print(json.dumps({k:v for k,v in out.items() if k!='previous_published'}, indent=2))
print('\nOLD (46-item test):', OLD)
print('saved detector_reeval.json / detector_reeval.csv')